# FLUXION — UI Mockup Generator (Batch)
> FLUX.1-schnell P100 | Output: PNG salvati in /kaggle/working/
> Dopo il run: `kaggle kernels output fluxiongestionale/fluxion-ai-generator`

In [ ]:
# CELLA 1 — INSTALL (P100 Python 3.12 — versioni pinned)
import subprocess, sys

# Pin preciso per evitare conflitti su Kaggle P100
packages = [
    'diffusers==0.32.2',
    'transformers==4.46.3',
    'accelerate==1.2.1',
    'sentencepiece',
    'pillow==10.4.0',
    'safetensors',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ Dipendenze installate!')


In [ ]:
# CELLA 2 — CARICA FLUX.1-schnell (float16, P100)
import torch
from diffusers import FluxPipeline

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.float16,
).to('cuda')
pipe.enable_attention_slicing()
print(f'✅ Modello caricato — VRAM usata: {torch.cuda.memory_allocated(0) / 1e9:.1f}GB')


In [ ]:
# CELLA 3 — GENERA MOCKUP UI FLUXION (tutte le schede)
import torch, os
from PIL import Image

OUT = '/kaggle/working/'
W, H, STEPS = 1280, 720, 6

def gen(prompt, filename, seed=42):
    print(f'⏳ {filename}...')
    g = torch.Generator(device='cuda').manual_seed(seed)
    result = pipe(prompt=prompt, width=W, height=H,
                  num_inference_steps=STEPS, generator=g, guidance_scale=0.0)
    img = result.images[0]
    img.save(f'{OUT}{filename}')
    print(f'  ✅ Salvata ({img.size[0]}x{img.size[1]})')
    return img

PROMPTS = {
    '01-scheda-parrucchiere.png': (
        'FLUXION Italian hair salon software, dark mode UI slate-900 background, '
        'client card showing haircut history timeline, color treatment records with '
        'hair color swatches, product allergy warnings in red badges, appointment '
        'list sidebar, teal-500 accent colors, glassmorphism cards, Italian labels '
        'Nome Cliente Servizio Data, professional desktop app 2026, photorealistic screenshot',
        11
    ),
    '02-scheda-fitness.png': (
        'FLUXION Italian gym management software, dark mode UI slate-900, '
        'fitness client profile card, circular BMI gauge indicator teal, '
        'body measurements chart with progress bars, workout plan weekly schedule, '
        'weight history line chart, body fat percentage metric cards, '
        'Italian labels Obiettivi Misurazioni Progressi, teal accent, '
        'professional sports management desktop app 2026, photorealistic UI screenshot',
        22
    ),
    '03-scheda-medica.png': (
        'FLUXION Italian medical clinic software, dark mode UI slate-900, '
        'patient medical record card, anamnesi section with medical history, '
        'current medications list with dosage, allergy warnings red badges, '
        'GDPR consent checkbox signed indicator green, appointment history, '
        'Italian labels Patologie Farmaci Allergie Consenso GDPR, '
        'clinical professional desktop app 2026, photorealistic UI screenshot',
        33
    ),
    '04-scheda-estetica.png': (
        'FLUXION Italian beauty salon software, dark mode UI slate-900, '
        'aesthetic client profile card, treatment history cosmetic procedures, '
        'skin type assessment cards, product preferences section, '
        'before-after photo placeholders, loyalty points badge teal, '
        'Italian labels Trattamenti Prodotti Preferenze Punti, '
        'luxury professional desktop app 2026, photorealistic UI screenshot',
        44
    ),
    '05-scheda-veicoli.png': (
        'FLUXION Italian auto repair shop software, dark mode UI slate-900, '
        'vehicle client card, car details targa modello anno, '
        'maintenance history timeline with service records, '
        'next service reminder alert orange badge, diagnostic notes section, '
        'parts replaced list, Italian labels Veicolo Manutenzione Prossima Revisione, '
        'professional workshop management desktop app 2026, photorealistic UI screenshot',
        55
    ),
    '06-dashboard-principale.png': (
        'FLUXION Italian PMI management software dashboard, dark mode slate-900, '
        'KPI cards revenue oggi settimana mese in teal, '
        'appointment calendar weekly view right side, '
        'sidebar navigation with icons Clienti Calendario Marketing Operatori, '
        'recent bookings list, quick action buttons, '
        'Italian labels professional desktop app 2026, photorealistic UI screenshot',
        66
    ),
    '07-calendario-prenotazioni.png': (
        'FLUXION Italian booking calendar software, dark mode UI slate-900, '
        'weekly calendar view with colored appointment cards, '
        'multiple staff columns each with color coding, '
        'time slots 8am to 8pm left sidebar, client names on appointment cards, '
        'service type labels, drag-drop visual indicators, '
        'teal-500 accent new appointment button top right, '
        'Italian professional desktop app 2026, photorealistic UI screenshot',
        77
    ),
}

results = []
for filename, (prompt, seed) in PROMPTS.items():
    img = gen(prompt, filename, seed)
    results.append(filename)

print(f'\n✅ DONE — {len(results)} immagini generate:')
for f in results:
    size = os.path.getsize(f'{OUT}{f}') // 1024
    print(f'  📸 {f} ({size}KB)')
